## Библиотеки и константы


In [18]:
import numpy as np

import random
import scipy.stats as stats
import pylab
import struct
import time
import os

In [19]:
n_substats = 4

TOTAL_COUNT = 1_000_000
BATCH_SIZE = 10_000
ARTIFACT_FORMAT = "<BBiBBiBBiBBi"
ARTIFACT_SIZE = struct.calcsize(ARTIFACT_FORMAT)

### Базовые значения

In [20]:
stats_map = {
    "HP%": 0,
    "HP": 1, 
    "ATK%": 2,
    "ATK": 3,
    "DEF%": 4,
    "DEF": 5,
    "EM": 6, # Elemental mastery
    "ER": 7, # Energy recharge
    "CRIT_RATE": 8, # Crit Rate
    "CRIT_DMG": 9, # Crit Damage
    "PHYS_DMG": 10,
    "PYRO_DMG": 11,
    "HYDRO_DMG": 12,
    "ANEMO_DMG": 13,
    "ELECTRO_DMG": 14,
    "DENDRO_DMG": 15,
    "CRYO_DMG": 16,
    "GEO_DMG": 17,
    "HEAL_BONUS": 18,
    "None": 100,
}

main_stat_values_map = {
    stats_map["HP%"]: 4660,
    stats_map["HP"]: 478000, 
    stats_map["ATK%"]: 4660,
    stats_map["ATK"]: 3110,
    stats_map["DEF%"]: 5830,
    stats_map["EM"]: 18700,
    stats_map["ER"]: 5180,
    stats_map["CRIT_RATE"]: 3110,
    stats_map["CRIT_DMG"]: 6220,
    stats_map["PYRO_DMG"]: 4660, # same for all elements
    stats_map["PHYS_DMG"]: 5830,
    stats_map["HEAL_BONUS"]: 3590
}


base_sub_stats_choices = [
    stats_map["HP%"],
    stats_map["HP"], 
    stats_map["ATK%"],
    stats_map["ATK"],
    stats_map["DEF%"],
    stats_map["DEF"],
    stats_map["EM"],
    stats_map["ER"],
    stats_map["CRIT_RATE"],
    stats_map["CRIT_DMG"]
]

# 100%, 90%, 80%, 70%
sub_stats_values_map = {
    stats_map["HP%"]: [408, 466, 525, 583],
    stats_map["HP"]: [20913, 23900, 26988, 29975], 
    stats_map["ATK%"]: [408, 466, 525, 583],
    stats_map["ATK"]: [1362, 1556, 1751, 1945],
    stats_map["DEF%"]: [510, 583, 656, 729],
    stats_map["DEF"]: [1620, 1852, 2083, 2315],
    stats_map["EM"]: [1632, 1865, 2098, 2331],
    stats_map["ER"]: [453, 518, 583, 648],
    stats_map["CRIT_RATE"]: [272, 311, 350, 389],
    stats_map["CRIT_DMG"]: [544, 622, 699, 777]
}

sub_stats_values_count = 4

sub_stats_weights_map = {
    stats_map["HP%"]: 4,
    stats_map["HP"]: 6,
    stats_map["ATK%"]: 4,
    stats_map["ATK"]: 6,
    stats_map["DEF%"]: 4,
    stats_map["DEF"]: 6,
    stats_map["EM"]: 4,
    stats_map["ER"]: 4,
    stats_map["CRIT_RATE"]: 3,
    stats_map["CRIT_DMG"]: 3
}

file_names = ["HP%", "HP", "ATK%", "ATK", "DEF%", "EM", "ER", "CRIT_RATE", "CRIT_DMG", "None"]

### Функции

In [25]:
def weighted_sample_no_replacement(population, weights):
    population = population.copy()
    weights = weights.copy()
    result = []
    for _ in range(n_substats):
        choice = random.choices(population, weights=weights, k=1)[0]
        idx = population.index(choice)
        result.append(choice)
        # убираем выбранный элемент
        population.pop(idx)
        weights.pop(idx)
    return result

def generate_art_for_file(population, weights, count_roll):
    artifact = [0] * 10
    stat_0_level_sample = weighted_sample_no_replacement(population, weights) # 4 sub stat in artifact
    stat_upgrade_choice = random.choices(stat_0_level_sample, k = count_roll) # rolls doring upgrade
    
    for stat in stat_0_level_sample:
        index = random.randint(0, sub_stats_values_count - 1)
        artifact[stat] += sub_stats_values_map[stat][index]
        
    for stat in stat_upgrade_choice:
        index = random.randint(0, sub_stats_values_count - 1)
        artifact[stat] += sub_stats_values_map[stat][index]
    
    return artifact

def generate_art_for_file_new(population, weights, count_roll):
    stat_0_level_sample = weighted_sample_no_replacement(population, weights)
    stat_upgrade_choice = random.choices(range(4), k=count_roll)

    artifact = []

    for stat in stat_0_level_sample:
        index = random.randint(0, sub_stats_values_count - 1)

        artifact.append([
            stat,
            1,
            sub_stats_values_map[stat][index]
        ])

    for stat_index in stat_upgrade_choice:
        stat = stat_0_level_sample[stat_index]

        index = random.randint(0, sub_stats_values_count - 1)

        artifact[stat_index][1] += 1
        artifact[stat_index][2] += sub_stats_values_map[stat][index]

    return artifact

def generate_file(deleted_stat_name):
    deleted_stat = stats_map[deleted_stat_name]
    possible_sub_stats = base_sub_stats_choices.copy()
    if deleted_stat in possible_sub_stats:
        possible_sub_stats.remove(deleted_stat)
    sub_stat_weights = [sub_stats_weights_map[value] for value in possible_sub_stats]
    with open(f"artifacts/artifacts_{deleted_stat_name}.bin", "wb") as file:

        count_3_base_stat_arts = TOTAL_COUNT * 4 // 5

        for start in range(0, count_3_base_stat_arts, BATCH_SIZE):
            artifacts = []
            for _ in range(BATCH_SIZE):
                artifacts.append(generate_art_for_file_new(possible_sub_stats, sub_stat_weights, 4))

            data = bytearray()
            # Запись batch в файл
            for artifact in artifacts:
                data.extend(
                    struct.pack(
                        ARTIFACT_FORMAT,
                        artifact[0][0], artifact[0][1], artifact[0][2],
                        artifact[1][0], artifact[1][1], artifact[1][2],
                        artifact[2][0], artifact[2][1], artifact[2][2],
                        artifact[3][0], artifact[3][1], artifact[3][2],
                    )
                )
            file.write(data)
            
            generated = start + BATCH_SIZE
            percent = generated / TOTAL_COUNT * 100

            print(
                f"\rGenerated {deleted_stat_name}: {generated:,} / {TOTAL_COUNT:,} "
                f"({percent:.2f}%)",
                end="",
                flush=True
            )
            
        for start in range(count_3_base_stat_arts, TOTAL_COUNT, BATCH_SIZE):
            artifacts = []
            for _ in range(BATCH_SIZE):
                artifacts.append(generate_art_for_file_new(possible_sub_stats, sub_stat_weights, 5))

            data = bytearray()
            # Запись batch в файл
            for artifact in artifacts:
                data.extend(
                    struct.pack(
                        ARTIFACT_FORMAT,
                        artifact[0][0], artifact[0][1], artifact[0][2],
                        artifact[1][0], artifact[1][1], artifact[1][2],
                        artifact[2][0], artifact[2][1], artifact[2][2],
                        artifact[3][0], artifact[3][1], artifact[3][2],
                    )
                )
            file.write(data)
            
            generated = start + BATCH_SIZE
            percent = generated / TOTAL_COUNT * 100
            
            print(
                f"\rGenerated {deleted_stat_name}: {generated:,} / {TOTAL_COUNT:,} "
                f"({percent:.2f}%)",
                end="",
                flush=True
            )

    print(f"\nGeneration {deleted_stat_name} completed.")

In [26]:
os.makedirs("artifacts", exist_ok=True)
start_time = time.perf_counter()
elapsed = 0

for main_stat_name in file_names:
    generate_file(main_stat_name)
    elapsed = time.perf_counter() - start_time
    print(f"--- {elapsed} seconds ---")

print(f"--- {elapsed} seconds of all artifacts ---")

Generated HP%: 1,000,000 / 1,000,000 (100.00%)
Generation HP% completed.
Generated HP: 1,000,000 / 1,000,000 (100.00%)
Generation HP completed.
Generated ATK%: 1,000,000 / 1,000,000 (100.00%)
Generation ATK% completed.
Generated ATK: 1,000,000 / 1,000,000 (100.00%)
Generation ATK completed.
Generated DEF%: 1,000,000 / 1,000,000 (100.00%)
Generation DEF% completed.
Generated EM: 1,000,000 / 1,000,000 (100.00%)
Generation EM completed.
Generated ER: 1,000,000 / 1,000,000 (100.00%)
Generation ER completed.
Generated CRIT_RATE: 1,000,000 / 1,000,000 (100.00%)
Generation CRIT_RATE completed.
Generated CRIT_DMG: 1,000,000 / 1,000,000 (100.00%)
Generation CRIT_DMG completed.
Generated None: 1,000,000 / 1,000,000 (100.00%)
Generation None completed.
--- 164.101060628891 seconds of all artifacts ---
